In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [3]:
# SET-UP

FUGLSANG_SUBJECTS = helper_functions.fuglsang_get_subjects()

In [ ]:
# ─────────────────────────────────────────────
# Run classifier — across dataset (Alice → Fuglsang)
# ─────────────────────────────────────────────

decisions_env   = {}
decisions_onset = {}
r_atts_env      = {}
r_igns_env      = {}
r_atts_onset    = {}
r_igns_onset    = {}

# Load Alice universal TRFs once
trf_env = eelbrain.load.unpickle(
    ALICE_TRF_DIR / f'64hz-universal-{helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD)}.pickle'
)
trf_onset = eelbrain.load.unpickle(
    ALICE_TRF_DIR / f'64hz-universal-{helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD)}.pickle'
)

for subject in FUGLSANG_SUBJECTS:
    eeg = eelbrain.load.unpickle(
        FUGLSANG_EEG_DIR / subject / f'{subject} eeg_trimmed.pickle'
    )
    true_att_env = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'attended_envelope_concat.pickle'
    ).x
    true_ign_env = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'ignored_envelope_concat.pickle'
    ).x
    true_att_onset = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'attended_envelope_onset_concat.pickle'
    ).x
    true_ign_onset = eelbrain.load.unpickle(
        FUGLSANG_PREDICTOR_DIR / subject / f'ignored_envelope_onset_concat.pickle'
    ).x

    print(trf_env)
    print(helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD))

    dec_env,   r_att_env,   r_ign_env   = helper_functions.aad_single_classifier(eeg, true_att_env,   true_ign_env,   trf_env)
    dec_onset, r_att_onset, r_ign_onset = helper_functions.aad_single_classifier(eeg, true_att_onset, true_ign_onset, trf_onset)

    decisions_env[subject]   = dec_env
    decisions_onset[subject] = dec_onset
    r_atts_env[subject]      = r_att_env
    r_igns_env[subject]      = r_ign_env
    r_atts_onset[subject]    = r_att_onset
    r_igns_onset[subject]    = r_ign_onset

    print(f"{subject}: env r_att={r_att_env:.3f}, r_ign={r_ign_env:.3f} | onset r_att={r_att_onset:.3f}, r_ign={r_ign_onset:.3f}")

acc_env   = sum(decisions_env.values())   / len(FUGLSANG_SUBJECTS)
acc_onset = sum(decisions_onset.values()) / len(FUGLSANG_SUBJECTS)
print(f"\n✅ Envelope classification: {acc_env:.2%}")
print(f"✅ Onset classification:    {acc_onset:.2%}")

# ─────────────────────────────────────────────
# Plot
# ─────────────────────────────────────────────

helper_functions.set_plot_style()

color_env   = 'tab:blue'
color_onset = 'tab:orange'
subjects    = list(r_atts_env.keys())

env_att   = [r_atts_env[s]   for s in subjects]
env_ign   = [r_igns_env[s]   for s in subjects]
onset_att = [r_atts_onset[s] for s in subjects]
onset_ign = [r_igns_onset[s] for s in subjects]

fig, ax = plt.subplots(figsize=(4.5, 4.5))

# Correct = circle, incorrect = X
for r_att, r_ign, decisions, color, marker, label in [
    (env_att,   env_ign,   decisions_env,   color_env,   'o', f'Envelope ({acc_env:.0%})'),
    (onset_att, onset_ign, decisions_onset, color_onset, '^', f'Onset ({acc_onset:.0%})'),
]:
    correct   = np.array([decisions[s] for s in subjects])
    incorrect = ~correct

    ax.scatter(np.array(r_att)[correct],   np.array(r_ign)[correct],
               color=color, marker=marker, s=70, zorder=4,
               label=f'{label} correct')
    ax.scatter(np.array(r_att)[incorrect], np.array(r_ign)[incorrect],
               color=color, marker='X', s=70, zorder=4,
               edgecolors='black', linewidths=0.8,
               label=f'{label} incorrect')

for s, xi, yi in zip(subjects, env_att, env_ign):
    ax.annotate(s, (xi, yi), textcoords='offset points',
                xytext=(5, 3), fontsize=7, color=color_env, alpha=0.85)
for s, xi, yi in zip(subjects, onset_att, onset_ign):
    ax.annotate(s, (xi, yi), textcoords='offset points',
                xytext=(5, 3), fontsize=7, color=color_onset, alpha=0.85)

all_vals = env_att + env_ign + onset_att + onset_ign
lim = max(all_vals) * 1.2
ax.plot([0, lim], [0, lim], color='gray', linestyle='--', linewidth=1,
        label=r'Chance ($r_{attended} = r_{ignored}$)', zorder=2)

ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel(r'$r_{\,attended}$', fontsize=12)
ax.set_ylabel(r'$r_{\,ignored}$', fontsize=12)
ax.set_title('Across-dataset (Alice → Fuglsang)', fontsize=12)
ax.legend(frameon=False, fontsize=8, loc='upper left')
ax.set_aspect('equal')

fig.tight_layout()
fig.savefig(FUGLSANG_FIGURES_DIR / 'across-dataset-scatter.png', dpi=150, bbox_inches='tight')
plt.show()

AttributeError: 'NDVar' object has no attribute 'h'